# BERT from scratch - FAN Yucheng

This embedding uses a four-layer BERT with hidden size 256, four attention heads, an FFN size of 1,024, a 256-token limit and an 8,000-token WordPiece vocabulary. Both tokenizer and encoder are trained from scratch; no pretrained model is downloaded. Training uses masked language modelling (MLM), without rating-label gradients or a classification head.

Each review produces a mean token vector from every Transformer layer, excluding padding, CLS and SEP. The four vectors are concatenated, L2-normalized, projected by a training-fitted PCA whitener to 512 features, and normalized again. A 1% scale floor limits amplification of low-variance directions. Independent analytics models consume this vector and the existing encoded metadata.

## Environment

Install `experiments/bert_requirements.txt` in the notebook's Python environment. It includes the unchanged shared `requirements.txt`. Use Python 3.12, PySpark 4.2.0 and Java 21. GPU execution needs a compatible PyTorch wheel. Installation details are in `src/embeddings/models/bert/environment_setup.md`.

## Training boundary

Keep the shared seed-42 train/validation/test split. BERT reserves about 10% of normalized-text groups inside training for fixed-mask MLM early stopping and checkpoint selection. The remaining training rows fit WordPiece, MLM and the projection. Grouping compares NFKC/HTML/case/whitespace-normalized text without changing the shared review column. Public validation and test are never read by `bert_train`.

All public training rows still receive vectors for the independent classifiers. Predict loads the frozen encoder, tokenizer and projection; it returns the shared narrow schema with a Spark ML `VectorUDT` column and explicit size metadata. The shared K-fold runner supplies one fitting row as its unused third frame.

## Run and retrain

`REBUILD=True` prepares shared inputs from the raw CSV. After that, matching inputs can use `False`. `FIT_ENCODER=True` calls training and reuses an already completed model only when its training configuration and corpus fingerprints match. `completion.json` marks a saved model complete; `training_config.json` and `input_manifest.json` are needed for this reuse check.

For a new training run, choose a fresh `MODEL_NAME` such as `bert_final`. Changing inputs or hyperparameters while keeping the existing model name raises an error rather than overwriting the saved model. An interrupted new run can use `RESUME=True` with identical inputs/settings and its local checkpoints. Optimizer checkpoints are not distributed with this completed model.

Train and predict share registry key `bert`. This personal runner uses local[4] and an 8g driver, and owns frame persistence and cache release through `DataOI`. Load saved vectors by `DataOI.load_embedding_data(MODEL_NAME)`. `RUN_TEAM_ANALYTICS=False` runs only BERT preparation and export. Set it to `True` to run the registered classifiers through the same shared interface. No classifier needs BERT's training-parameter dictionary.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "src").is_dir() and (candidate / "project.ipynb").is_file():
        ROOT = candidate
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pyspark.sql import SparkSession
from src.dataOI import DataOI

SEED = 42

# True: read the raw CSV, then clean, engineer text, split, encode, and save that split.
# False: skip those steps and load the saved train, validation, and test frames from data/processed/split.
REBUILD = True

# `spark.driver.memory` is read only when the JVM starts. Restart the kernel after changing it.
spark = SparkSession.builder \
                            .master("local[4]")\
                            .appName("DrugReviewClassification")\
                            .config("spark.driver.memory","8g")\
                            .getOrCreate()
spark_context = spark.sparkContext
spark_context.setLogLevel("ERROR")
data_oi = DataOI()


In [ ]:
from src.cleaning import data_cleaning
from src.text_features import text_feature_engineering
from src.encoder import encoder
from src.split import split, weight

if REBUILD:
    df_raw = data_oi.load_raw_data()
    df_cleaned = data_cleaning(df_raw).cache()
    df_cleaned.count()
    df_engineered = text_feature_engineering(df_cleaned, True).cache()
    df_engineered.count()
    data_oi.unpersist(df_cleaned)
    del df_raw, df_cleaned
    split_frames = split(df_engineered, SEED)
    df_weighted = weight(split_frames[0])
    df_train, df_valid, df_test = encoder(df_weighted, *split_frames[1:])
    for frame in (df_train, df_valid, df_test):
        frame.count()
    data_oi.save_split_data(df_train, df_valid, df_test)
    data_oi.unpersist(df_engineered, df_weighted, *split_frames)
    del df_engineered, df_weighted, split_frames, frame
else:
    df_train, df_valid, df_test = data_oi.load_split_data()


In [ ]:
from src.embeddings import EmbeddingModel
from src.embeddings.bert import BertTrainingConfig

embedding_name = "bert"
MODEL_NAME = "bert"
FIT_ENCODER = True
RUN_TEAM_ANALYTICS = False
RESUME = False
ANALYTICS_PARAMETERS = {}  # Model name -> flat constructor-parameter dict.
BERT_CONFIG = BertTrainingConfig()
# For a fresh run only: BERT_CONFIG = BertTrainingConfig(batch_size=64, gradient_accumulation_steps=2)

# BERT reads full reviews; the optional phrase frames are not needed here.
embedding_model = EmbeddingModel(df_train, df_valid, df_test, phrases=None)
if FIT_ENCODER:
    EmbeddingModel.train_registry[embedding_name](
        embedding_model, model_name=MODEL_NAME, config=BERT_CONFIG,
        resume=RESUME,
    )


In [ ]:
print(f"Embedding Model `{embedding_name}` Running")
train_embedding, valid_embedding, test_embedding = EmbeddingModel.predict_registry[embedding_name](
    embedding_model, MODEL_NAME)
data_oi.save_embedding_data(MODEL_NAME, train_embedding, valid_embedding, test_embedding)
data_oi.unpersist(train_embedding, valid_embedding, test_embedding)
del train_embedding, valid_embedding, test_embedding


In [ ]:
data_oi.unpersist(df_train, df_valid, df_test)
del embedding_model, df_train, df_valid, df_test


In [ ]:
if RUN_TEAM_ANALYTICS:
    from src.analytics import AnalyticsModel

    train_embedding, valid_embedding, test_embedding = data_oi.load_embedding_data(MODEL_NAME)
    analytics_model = AnalyticsModel(MODEL_NAME, train_embedding, valid_embedding, test_embedding)


In [ ]:
if RUN_TEAM_ANALYTICS:
    run_names = []
    try:
        for name, function in AnalyticsModel.registry.items():
            print(f"Embedding Model `{embedding_name}` Analytics Model `{name}` Running")
            model_name = f"{name}__{MODEL_NAME}"
            try:
                if name in ANALYTICS_PARAMETERS:
                    predictions = function(analytics_model, ANALYTICS_PARAMETERS[name])
                else:
                    predictions = function(analytics_model)
                data_oi.save_prediction_data(model_name, *predictions)
                run_names.append(model_name)
                del predictions
            finally:
                analytics_model.release_features()
    finally:
        data_oi.unpersist(train_embedding, valid_embedding, test_embedding)
        del train_embedding, valid_embedding, test_embedding, analytics_model


In [ ]:
if RUN_TEAM_ANALYTICS:
    from src.evaluation import Scorer

    train_predictions, valid_predictions, test_predictions = data_oi.load_prediction_data(run_names)
    for label, predictions in [("Training", train_predictions),
                               ("Validation", valid_predictions), ("Testing", test_predictions)]:
        print(f"{label} Data:")
        Scorer({name: predictions[name] for name in run_names}).show_metrics(truncate=80)
